# Fresh Data — Veille Technologique OCR & DocVQA

> - Famille 1 : **VLM/DocVQA** (LayoutLMv3, LayoutLLM, Pix2Struct-like)
> - Famille 2 : **OCR classiques** + benchmarks (EasyOCR/PaddleOCR, CC-OCR/ICDAR/MP-DocVQA)

## Contenu du notebook
1. Pré-requis & installation
2. Données d’exemple (étiquettes)
3. OCR « classiques » (EasyOCR, PaddleOCR) — extraction brute
4. Post-traitement : regex & parsing sémantique (Origine, Espèce, Calibre, Lot, GGN)
5. DocVQA (LayoutLMv3) — Q/R sur document/étiquette
6. Évaluation rapide & recommandations pour Fresh Data



## 1) Pré-requis & installation

Exécutez une seule fois :


> Si pas de GPU, installe PyTorch CPU (`--index-url` non nécessaire).


In [2]:
# OCR
%pip install easyocr paddlepaddle paddleocr

# Vision/HF (DocVQA)
%pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121  # (ou +cpu)
%pip install transformers==4.44.2 accelerate==0.34.2 datasets pillow

# Outils
%pip install matplotlib opencv-python rapidfuzz python-Levenshtein

  Using cached chardet-5.2.0-py3-none-any.whl.metadata (3.4 kB)
  Using cached numpy-2.2.6-cp310-cp310-win_amd64.whl.metadata (60 kB)
  Using cached lazy_loader-0.4-py3-none-any.whl.metadata (7.6 kB)
  Using cached typing_extensions-4.15.0-py3-none-any.whl.metadata (3.3 kB)
   ---------------------------------------- 0.0/2.9 MB ? eta -:--:--
   ---------------------------------------- 2.9/2.9 MB 18.5 MB/s eta 0:00:00
   ---------------------------------------- 0.0/101.7 MB ? eta -:--:--
   -- ------------------------------------- 6.8/101.7 MB 35.0 MB/s eta 0:00:03
   ----- ---------------------------------- 14.4/101.7 MB 36.3 MB/s eta 0:00:03
   -------- ------------------------------- 22.0/101.7 MB 36.7 MB/s eta 0:00:03
   ----------- ---------------------------- 29.6/101.7 MB 36.2 MB/s eta 0:00:02
   -------------- ------------------------- 37.0/101.7 MB 36.2 MB/s eta 0:00:02
   ----------------- ---------------------- 44.3/101.7 MB 35.7 MB/s eta 0:00:02
   -------------------- -----

  You can safely remove it manually.
  You can safely remove it manually.
ERROR: Could not install packages due to an OSError: [WinError 5] Accès refusé: 'c:\\Users\\pc\\anaconda3\\envs\\projet\\Lib\\site-packages\\cv2\\cv2.pyd'
Consider using the `--user` option or check the permissions.



Note: you may need to restart the kernel to use updated packages.


ERROR: Invalid requirement: '#': Expected package name at the start of dependency specifier
    #
    ^


  Using cached httpx-0.28.1-py3-none-any.whl.metadata (7.1 kB)
  Using cached anyio-4.12.0-py3-none-any.whl.metadata (4.3 kB)
  Using cached httpcore-1.0.9-py3-none-any.whl.metadata (21 kB)
  Using cached aiohappyeyeballs-2.6.1-py3-none-any.whl.metadata (5.9 kB)
  Using cached async_timeout-5.0.1-py3-none-any.whl.metadata (5.1 kB)
   ---------------------------------------- 0.0/9.5 MB ? eta -:--:--
   --------------------------- ------------ 6.6/9.5 MB 36.6 MB/s eta 0:00:01
   ---------------------------------------- 9.5/9.5 MB 29.5 MB/s eta 0:00:00
Using cached httpx-0.28.1-py3-none-any.whl (73 kB)
Using cached httpcore-1.0.9-py3-none-any.whl (78 kB)
   ---------------------------------------- 0.0/28.1 MB ? eta -:--:--
   ---------- ----------------------------- 7.6/28.1 MB 36.0 MB/s eta 0:00:01
   -------------------- ------------------- 14.4/28.1 MB 34.8 MB/s eta 0:00:01
   ----------------------------- ---------- 20.4/28.1 MB 32.3 MB/s eta 0:00:01
   -------------------------------


## 2) Données d’exemple (étiquettes)

Dépose les images dans un dossier `data/` à côté du notebook

La cellule ci-dessous liste les images trouvées.


In [3]:

import os, glob
from pathlib import Path

SEARCH_PATHS = [
    'data/*.png','data/*.jpg','data/*.jpeg',
    '/mnt/data/*.png','/mnt/data/*.jpg','/mnt/data/*.jpeg'
]

images = []
for pat in SEARCH_PATHS:
    images.extend(glob.glob(pat))

print(f'Images détectées ({len(images)}):')
for i, p in enumerate(images[:10], 1):
    print(f'{i:02d} - {p}')


Images détectées (655):
01 - data\20251030_144608.jpg
02 - data\20251030_144840.jpg
03 - data\20251030_144851.jpg
04 - data\20251030_144906.jpg
05 - data\20251030_144938.jpg
06 - data\20251030_144953.jpg
07 - data\20251030_145011.jpg
08 - data\20251030_145018.jpg
09 - data\20251030_145031.jpg
10 - data\20251030_145040.jpg



## 3) OCR « classiques » — EasyOCR

**Quand l’utiliser ?**  
- Photos réelles (basse résolution, angle, reflets).  
- Lecture brute du texte d’une étiquette.

**Forces** : simple, rapide, multi-langues.  
**Limites** : ne comprend pas la sémantique; nécessite post-traitement.

### Exemple : OCR d'une étiquette


In [6]:
# --- EasyOCR with EXIF orientation fix (consistent display + OCR) ---

import glob, cv2, easyocr, numpy as np
from PIL import Image, ImageDraw, ImageOps
from pathlib import Path

def load_image_consistent(path):
    """Load once, fix EXIF orientation, return (pil_rgb, cv_bgr)."""
    pil = Image.open(path)
    pil = ImageOps.exif_transpose(pil)   # <- honors EXIF and applies the rotation to pixels
    pil = pil.convert("RGB")
    cv  = cv2.cvtColor(np.array(pil), cv2.COLOR_RGB2BGR)  # same orientation in OpenCV
    return pil, cv

def clip_box(box, w, h):
    return [[max(0, min(w-1, int(x))), max(0, min(h-1, int(y)))] for x, y in box]

# 1) Find images
images = sorted(glob.glob("data/*.jpg") + glob.glob("data/*.jpeg") + glob.glob("data/*.png"))
print(f"Images détectées : {len(images)}")
if not images:
    raise SystemExit("Aucune image dans data/*.jpg|*.png")

# 2) Pick one
image_path = images[6]  # ou 'data/20251030_144840.jpg'
print("Image utilisée :", image_path)

# 3) Load once, EXIF-normalized
pil_img, cv_img = load_image_consistent(image_path)
h, w = cv_img.shape[:2]
print(f"Dimensions : {w}x{h}")

# 4) OCR on the normalized pixels
reader = easyocr.Reader(['fr','en'], gpu=False)
result = reader.readtext(
    cv_img,
    detail=1,
    paragraph=False,
    contrast_ths=0.1,
    adjust_contrast=0.7,
    decoder='greedy',
    min_size=10,
)

# 5) Clean bboxes (avoid zero-size crops)
clean = []
for (bbox, text, conf) in result:
    bbox = clip_box(bbox, w, h)
    xs = [p[0] for p in bbox]; ys = [p[1] for p in bbox]
    if (max(xs)-min(xs) > 2) and (max(ys)-min(ys) > 2):
        clean.append((bbox, text, conf))
result = clean

# 6) Print text
print("\n--- Texte détecté ---")
for (_, text, conf) in result:
    print(f"- {text} (conf={conf:.2f})")

# 7) Draw boxes on the same EXIF-fixed PIL image
draw = ImageDraw.Draw(pil_img)
for (bbox, text, conf) in result:
    xs = [p[0] for p in bbox]; ys = [p[1] for p in bbox]
    draw.rectangle([min(xs),min(ys),max(xs),max(ys)], outline=(255,0,0), width=2)

# 8) Save without EXIF (so it stays upright everywhere)
out_dir = Path("out_boxes"); out_dir.mkdir(parents=True, exist_ok=True)
boxed_path = out_dir / (Path(image_path).stem + "_boxed.jpg")
pil_img.save(boxed_path)  # EXIF is dropped by default
print("\n✅ Image annotée :", boxed_path)

# show in notebook
pil_img


ModuleNotFoundError: No module named 'easyocr'


## 4) Post-traitement : Réorganisation et post-traitement sémantique des résultats OCR


Ce bloc sert à **reclasser, nettoyer et interpréter** les textes détectés par EasyOCR afin de les rendre exploitables dans le cadre du projet **Fresh Data**.  
L’objectif est de transformer un résultat OCR brut — souvent désordonné et bruité — en un **ensemble d’informations structurées** correspondant aux mentions réglementaires d’une étiquette (espèce, origine, calibre, etc.).

---

### Étapes réalisées

1. **Tri et réorganisation des tokens OCR**  
   → Les zones de texte détectées sont triées du haut vers le bas puis de la gauche vers la droite.  
   → Cela rétablit un ordre de lecture “humain”, même si EasyOCR renvoie les mots dans le désordre.

2. **Regroupement par lignes et fusion du texte**  
   → Les mots proches verticalement sont fusionnés sur une même ligne.  
   → Le script corrige les séparateurs et erreurs typiques d’OCR :  
     - `.spb1` devient `LOT: spb1`  
     - `CATI` devient `CAT: I`  
     - `CAL . 136/165` devient `CALIBRE: 136/165`  
     - `CATÉOORIE` devient `CATEGORIE`

3. **Nettoyage et harmonisation**  
   → Tous les libellés sont normalisés (accents, majuscules, espaces).  
   → Les champs ambigus (`CAT`, `CAL`, `LOT`, etc.) sont reformulés pour correspondre aux libellés attendus.  
   → Le code gère aussi les cas où plusieurs mentions se retrouvent sur la même ligne (`ORIGINE: ESPAGNE CATÉGORIE: EXTRA`).

4. **Extraction sémantique avec expressions régulières (Regex)**  
   → Une série de règles détecte et extrait automatiquement les informations clés :  
     - **Espèce / variété** (ex. *COX’S ORANGE*, *POMELO*)  
     - **Origine** (ex. *FRANCE*, *ESPAGNE*)  
     - **Calibre** (ex. *136/165*, *70-80*)  
     - **Catégorie** (ex. *CAT I*, *EXTRA*)  
     - **Lot** (ex. *spb1*)  
     - **GGN** (numéro global de traçabilité, 10 à 14 chiffres)  
     - **Date d’emballage** (ex. *J 13*)

   **Exemples de motifs utilisés :**
   - `Origine\s*[:\-]?\s*([A-Za-zÀ-ÿ\s]+)`  
   - `Calibre\s*[:\-]?\s*([0-9xX/ \-–]+)`  
   - `Cat(?:é|e)gorie\s*[:\-]?\s*([0-9A-Za-z]+)`  
   - `Lot\s*[:\-]?\s*([A-Za-z0-9\-_.]+)`  
   - `GGN\s*[:\-]?\s*([0-9]{10,14})`

5. **Affichage des résultats**  
   - La section `--- LECTURE ORDONNÉE (lignes) ---` montre le texte relu dans l’ordre logique (comme un humain lirait l’étiquette).  
   - La section `--- CHAMPS EXTRAIts ---` affiche les champs clés extraits et normalisés.

---

### Objectif pour Fresh Data

Ce traitement constitue une étape essentielle du pipeline :
- Il **structure automatiquement** les informations issues d’une photo d’étiquette réelle.  
- Il permet la **comparaison automatique** entre les données détectées et les documents BL/BC.  
- Il facilite la **détection d’écarts** (origine, calibre, lot) et renforce la **traçabilité produit par produit** conformément à la **réglementation européenne 2025**.

En résumé, ce bloc transforme une simple image brute en un **jeu de données exploitable** pour la vérification de conformité et la fiabilisation des flux logistiques dans le secteur des fruits et légumes.



In [4]:
import re, unicodedata, numpy as np

LABELS_ANCHORS = r"(?:CATEGORIE|CAT|CALIBRE|CAL|LOT|GGN|POIDS|VARI(?:E|É)TE|ORIGINE|DATE|TRAC|GGN|CODE)"

def _normalize_token(t: str) -> str:
    t = unicodedata.normalize("NFKC", t)
    t = t.replace("’", "'").replace("`", "'")
    t = re.sub(r"\s+", " ", t)
    return t.strip()

def _canonize_labels(text: str) -> str:
    """Corrige les fautes OCR fréquentes et unifie les libellés."""
    # Corrige les répétitions et erreurs typiques
    text = re.sub(r'(.)\1{2,}', r'\1', text)
    text = re.sub(r'CAT[ÉE]O+RIE', 'CATEGORIE', text, flags=re.IGNORECASE)
    text = re.sub(r'CATE?GORIE', 'CATEGORIE', text, flags=re.IGNORECASE)
    text = re.sub(r'CATEGORIE[^:]', 'CATEGORIE: ', text, flags=re.IGNORECASE)
    text = re.sub(r'CATI', 'CATEGORIE: I', text, flags=re.IGNORECASE)
    text = re.sub(r'CAL[ .:]?IBRE', 'CALIBRE', text, flags=re.IGNORECASE)
    text = re.sub(r'CALIBRE[^:]', 'CALIBRE: ', text, flags=re.IGNORECASE)
    text = re.sub(r'CAL\s*[:;.\-]?\s*', 'CALIBRE: ', text, flags=re.IGNORECASE)
    text = re.sub(r'LOT\s*[:;.\-]?\s*', 'LOT: ', text, flags=re.IGNORECASE)
    text = re.sub(r'ORIGI?NE\s*[:;.\-]?\s*', 'ORIGINE: ', text, flags=re.IGNORECASE)
    text = re.sub(r'VARI[EÉ]T[EÉ]\s*[:;.\-]?\s*', 'VARIETE: ', text, flags=re.IGNORECASE)
    text = re.sub(r'CATÉOORIE', 'CATEGORIE', text, flags=re.IGNORECASE)
    text = re.sub(r'EGORIE', 'CATEGORIE', text, flags=re.IGNORECASE)
    text = re.sub(r'CATEGORIE: CATEGORIE', 'CATEGORIE', text, flags=re.IGNORECASE)
    text = re.sub(r'\s+\.\s+', ' : ', text)
    text = re.sub(r'\s+', ' ', text)
    return text.strip()

def _result_to_lines(result):
    """Recompose les lignes dans l'ordre naturel haut->bas puis gauche->droite."""
    items = []
    for (bbox, text, conf) in result:
        xs = [p[0] for p in bbox]; ys = [p[1] for p in bbox]
        w, h = max(xs)-min(xs), max(ys)-min(ys)
        if w <= 2 or h <= 2: continue
        items.append({"x":min(xs),"y":min(ys),"xc":np.mean(xs),"yc":np.mean(ys),"w":w,"h":h,"text":_normalize_token(text),"conf":conf})
    if not items: return []
    items.sort(key=lambda t:(t["yc"],t["x"]))
    median_h = np.median([it["h"] for it in items]) or 12
    row_thresh = 0.6 * median_h
    rows = []
    for it in items:
        for row in rows:
            if abs(it["yc"]-row["yc"])<=row_thresh:
                row["items"].append(it)
                break
        else:
            rows.append({"yc":it["yc"],"items":[it]})
    for row in rows: row["items"].sort(key=lambda t:t["x"])
    lines = []
    for row in rows:
        line = " ".join(it["text"] for it in row["items"])
        line = _canonize_labels(line)
        lines.append(line.strip())
    return lines

def parse_fields_from_lines(lines):
    """Extraction sémantique tolérante aux fautes OCR (séparateurs, accents, doublons)."""
    blob = _canonize_labels("\n".join(lines))

    # Ajout de normalisations simples
    blob = blob.replace("CAT;", "CATEGORIE: ")
    blob = re.sub(r"\bCAT\s*[;., ]+\s*([0-9I]+)\b", r"CATEGORIE: \1", blob, flags=re.IGNORECASE)
    blob = re.sub(r"(\d{2,3}[-/xX]\d{2,3}\s*[Gg]?)", r"CALIBRE: \1", blob, flags=re.IGNORECASE)
    blob = blob.replace("ORIGINE :", "ORIGINE:")  # espace après ORIGINE

    def grab(pat):
        m = re.search(pat, blob, flags=re.IGNORECASE)
        return (m.group(1).strip() if m else "")

    # ORIGINE: FRANCE
    origine = grab(r"ORIGINE[:\-]?\s*([A-ZÀ-ÿ\s]+)")
    origine = origine.title()

    # ESPECE / VARIETE
    espece = grab(r"\b(COX['’]S ORANGE|GALA|POMMES?|POIRES?|BANANE|TOMATE|ORANGE|CITRON|RAISIN|CAROTTE|SALADE|POMME DE TERRE)\b")

    # CALIBRE: 201-240G / 70-80 / 136/165
    calibre = grab(r"CALIBRE[:\-]?\s*([0-9xX/ \-–]{2,8}[Gg]?)")

    # CATEGORIE: EXTRA / I / II / 1
    categorie = grab(r"CATEGORIE[:\-;]?\s*([A-Za-z0-9ÉI]+)")
    categorie = categorie.replace("É", "E").upper()
    if categorie in {"1","I"}: categorie = "I"
    elif categorie in {"2","II"}: categorie = "II"
    elif categorie in {"3","III"}: categorie = "III"

    # LOT
    lot = grab(r"LOT[:\-]?\s*([A-Za-z0-9\-_.]+)")

    # GGN
    ggn = grab(r"GGN[:\-]?\s*([0-9][0-9\s]{9,16}[0-9])")
    if ggn:
        ggn = re.sub(r"\s+","",ggn)

    # DATE D'EMBALLAGE J 13
    m = re.search(r"DATE D['’]EMBALLAGE.*?\b([A-Z])\s*([0-9]{1,2})\b", blob, re.IGNORECASE)
    date_emballage = f"{m.group(1)} {m.group(2)}" if m else ""

    return {
        "espece": espece,
        "origine": origine,
        "calibre": calibre,
        "categorie": categorie,
        "lot": lot,
        "ggn": ggn,
        "date_emballage": date_emballage,
        "blob_debug": blob
    }


def parse_fields(texts_or_result):
    if isinstance(texts_or_result, list) and (not texts_or_result or isinstance(texts_or_result[0], str)):
        lines = [_canonize_labels(_normalize_token(t)) for t in texts_or_result]
    else:
        lines = _result_to_lines(texts_or_result)
    return parse_fields_from_lines(lines)

# Affichage des lignes reconstituées

parsed = parse_fields(result)
parsed


{'espece': "COX'S ORANGE",
 'origine': 'France Cat',
 'calibre': '136/165',
 'categorie': '',
 'lot': 'spb1',
 'ggn': '',
 'date_emballage': '',
 'blob_debug': "COX'S ORANGE CALIBRE: IBRE: CALIBRE: 136/165 ORIGINE: FRANCE CAT:I Guillaume et Mathieu 1 BOLLART 59 630 tel '508 PiesosRouck 9 DATE DEMBALLAGE J 13 LOT: spb1"}


## 4-bis) Comparaison avec BC/BL

Supposons qu'on charge un **BL** au format JSON (ou Excel → JSON) contenant les champs attendus.  
On compare par **égalité** ou **similarité** (distance de Levenshtein via `rapidfuzz`).



In [39]:

from rapidfuzz import fuzz

# Exemple attendu (depuis un BL parsé)
expected = {
    'espece': 'Orange',
    'origine': 'France',
    'calibre': '136/165',
    'categorie': '1',
    'lot': 'spb 1',
}

def compare_dicts(pred, exp):
    report = {}
    keys = set(pred) | set(exp)
    for k in keys:
        pv, ev = pred.get(k,''), exp.get(k,'')
        if pv=='' or ev=='':
            score = 0 if ev and not pv else 100 if not ev and not pv else 0
        else:
            score = fuzz.WRatio(str(pv), str(ev))
        report[k] = {'pred': pv, 'expected': ev, 'match_score': score}
    return report

report = compare_dicts(parsed, expected)
report


{'calibre': {'pred': '136/165', 'expected': '136/165', 'match_score': 100.0},
 'espece': {'pred': "COX'S ORANGE", 'expected': 'Orange', 'match_score': 22.5},
 'date_emballage': {'pred': '', 'expected': '', 'match_score': 100},
 'ggn': {'pred': '', 'expected': '', 'match_score': 100},
 'categorie': {'pred': '', 'expected': '1', 'match_score': 0},
 'blob_debug': {'pred': "COX'S ORANGE CALIBRE: IBRE: CALIBRE: 136/165 ORIGINE: FRANCE CAT:I Guillaume et Mathieu 1 BOLLART 59 630 tel '508 PiesosRouck 9 DATE DEMBALLAGE J 13 LOT: spb1",
  'expected': '',
  'match_score': 0},
 'lot': {'pred': 'spb1',
  'expected': 'spb 1',
  'match_score': 88.88888888888889},
 'origine': {'pred': 'France Cat', 'expected': 'France', 'match_score': 90.0}}

## 5) DocVQA — Pix2Struct (QA image → texte, sans Tesseract)

**Principe.** Pix2Struct (*google/pix2struct-docvqa-base*) prend une **image** et une **question** et génère directement une **réponse textuelle**.  
Contrairement à LayoutLMv3, il **n’utilise pas Tesseract** : pas d’OCR externe à installer. Idéal pour poser des questions simples sur une **étiquette nette** (ex. *« Quelle est l’origine ? »*, *« Quel est le lot ? »*).

**Forces :**  
- Dépendances légères (pas de Tesseract) ; facile à lancer.  
- Très pratique pour **valider** un champ déjà détecté par OCR.

**Limites :**  
- Moins robuste que l’OCR sur photos **très bruitées/penchées**.  
- Ce n’est pas un lecteur de texte exhaustif comme EasyOCR ; pense-le comme un **outil de vérification ciblée** (Q/R).

**Recommandation Fresh Data :**  
- Utiliser **EasyOCR + parsing** pour **lire** (terrain), puis **Pix2Struct** pour **valider** 1–2 champs clés (ex. *origine*, *lot*, *calibre*) quand l’étiquette est suffisamment lisible.


In [ ]:
# --- DocVQA avec Pix2Struct : bloc de test prêt à l’emploi ---
# (si nécessaire) pip install -U transformers pillow torch --extra-index-url https://download.pytorch.org/whl/cpu
# Sur GPU: installe torch avec CUDA approprié

from transformers import Pix2StructForConditionalGeneration, Pix2StructProcessor
from PIL import Image, ImageOps
import torch, glob, os

# 1) Choisir une image d’étiquette (prend la première du dossier data/)
img_paths = sorted(glob.glob("data/*.jpg") + glob.glob("data/*.png"))
assert img_paths, "Aucune image trouvée dans data/*.jpg|*.png"
img_path = img_paths[22]
print("Image utilisée :", img_path)

# 2) Charger image et normaliser orientation EXIF (affichage cohérent)
image = Image.open(img_path)
image = ImageOps.exif_transpose(image).convert("RGB")

# (Option) réduire la taille si la photo est géante pour accélérer l’inférence
max_side = 1280
if max(image.size) > max_side:
    ratio = max_side / max(image.size)
    image = image.resize((int(image.width*ratio), int(image.height*ratio)))

# 3) Charger modèle + processor
device = "cuda" if torch.cuda.is_available() else "cpu"
processor = Pix2StructProcessor.from_pretrained("google/pix2struct-docvqa-base")
model = Pix2StructForConditionalGeneration.from_pretrained("google/pix2struct-docvqa-base").to(device).eval()

def docvqa_answer(question: str, max_new_tokens: int = 32) -> str:
    """Pose une question à l’image, renvoie la réponse générée."""
    inputs = processor(images=image, text=question, return_tensors="pt").to(device)
    with torch.no_grad():
        out_ids = model.generate(**inputs, max_new_tokens=max_new_tokens)
    return processor.decode(out_ids[0], skip_special_tokens=True).strip()

# 4) Questions typiques pour nos étiquettes F&L
questions = [
    "Quelle est l'origine ?",
    "Quel est le calibre ?",
    "Quel est le lot ?",
    "Quelle est la catégorie ?",
    "Quelle est l'espèce ou variété ?",
    "Quelle est le calibre ?"
]

print("\n--- DocVQA (Pix2Struct) ---")
for q in questions:
    ans = docvqa_answer(q)
    print(f"{q} → {ans if ans else '(aucune réponse)'}")

# Conseils d’usage :
# - Si la réponse est vide/inexacte, teste un libellé plus simple (ex. "Origine ?").
# - Pix2Struct marche mieux si l’étiquette est cadrée (tu peux cropper la zone d’étiquette avant).
# - Pense ce modèle comme un VALIDATEUR rapide, en complément de l’OCR terrain.


Image utilisée : data\20251030_150250.jpg

--- DocVQA (Pix2Struct) ---
Quelle est l'origine ? → FRANCE
Quel est le calibre ? → ?
Quel est le lot ? → E.A.R.L. du Petit Pavilion
Quelle est la catégorie ? → Pommes de table
Quelle est l'espèce ou variété ? → POMMES DE TABLE
Quelle est le calibre ? → ?


## 6) Évaluation rapide — OCR & DocVQA (terrain Fresh Data)

### Portée du notebook
- **OCR terrain (EasyOCR)** + correctif **EXIF** + **réordonnancement** des tokens.
- **Extraction sémantique** des champs clés (Espèce/Variété, Origine, Calibre, Catégorie, Lot, GGN, Date).
- **Vérification DocVQA (Pix2Struct)** pour **valider** 1–2 champs sur étiquette nette.
- **Comparaison BL/BC** (similarité par champ, RapidFuzz).

### Constats sur nos images (échantillon data/)
- **Lecture brute** : bonne sur textes gras et libellés (“ORIGINE”, “LOT”), plus fragile sur **petits caractères** (GGN, lignes d’adresse).  
- **Erreurs typiques** rencontrées et partiellement corrigées :
  - séparateurs `CAT; 1` / `LOT .spb1` → normalisés en `CATEGORIE: I` / `LOT: spb1`;
  - variantes bruitées `CATÉOORIE`/`CAl: IBRE` → remises en `CATEGORIE`/`CALIBRE`.
- **DocVQA (Pix2Struct)** :
  - utile comme **contrôle ciblé** (“Quelle est l’origine ?”), mais ne remplace pas la lecture exhaustive sur photo bruitée.

### Utilité par famille (résumé)
| Famille | Rôle dans Fresh Data | Verdict |
|---|---|---|
| **EasyOCR/PaddleOCR** | Lecture **terrain** (photos réelles) | ✅ Pilier de la lecture |
| **Post-traitement (règles/regex)** | Structuration & normalisation (lot, calibre, cat., origine) | ✅ Indispensable |
| **DocVQA (Pix2Struct)** | **Validation** ponctuelle de champs sur étiquette nette | ✅ Utile en complément |
| **LayoutLMv3/LLM** | QA/structure sur doc propre (BL/BC, PDF) | ⚠️ À réserver (lourd, pas terrain) |
| **Benchmarks CC-OCR/ICDAR/MP-DocVQA** | **Mesure** de robustesse (pas inférence) | ✅ À utiliser pour évaluer |

### KPI à suivre (et seuils proposés pour le POC)
- **Taux d’extraction par champ** (sur 100 photos terrain)  
  - Origine ≥ **95%**, Lot ≥ **90%**, Calibre ≥ **90%**, Catégorie ≥ **92%**, GGN ≥ **85%**.  
- **Taux d’accord BL/BC** (exact/similaire via RapidFuzz) ≥ **95%**.  
- **Latence mobile** (capturer → résultat) : **< 2,0 s** pour 1 étiquette (Edge/CPU).  
- **Robustesse** (rotation ±10°, flou léger) : dégradation **< 10 pts** sur les KPI ci-dessus.

### Risques & contournements
- **Bruit visuel / fond bois / agrafes** → Ajouter **crop d’étiquette (YOLO)** avant OCR.  
- **Petits glyphes (GGN)** → Sur-échantillonnage local, **sharpen** léger avant OCR; fallback DocVQA si étiquette nette.  
- **Étiquettes partiellement masquées** → multi-prise (2 photos) + règles de fusion.  
- **EXIF / orientation** → déjà corrigé (EXIF transpose) ; garder ce pré-traitement.

### Actions restantes pour la démo client
1. **Ajouter batch → CSV** (toutes les images `data/` → `results_ocr.csv` avec textes + champs).  
2. **Ajouter mini-bench “robustesse”** (rotation/flou) et reporter les KPI ci-dessus.  
3. **(Option) Détection YOLO** pour crop d’étiquette (réduction des faux positifs et meilleure lecture GGN).  
4. **Passer aux helpers `parse_fields_v2`** (tolérance renforcée) pour stabiliser Catégorie/Calibre/Lot.

> **Conclusion** — La pile **EasyOCR + parsing** couvre l’usage **terrain** attendu et, couplée à **Pix2Struct** en validation ciblée, répond aux besoins du POC (traçabilité par produit). Les compléments proposés (batch/bench/crop) permettent de **mesurer** et **sécuriser** les performances avant terrain réel.
